In [3]:
library(readr)
library(dplyr)
library(janitor)
library(stringr)
library(tidyr)
library(stringdist)

In [4]:
scraped <- read_csv("data/brfss_export_data/brfss_2014_phr_1_8_11.csv") |> clean_names()
brfss <- read_csv("data/brfss_all_categories.csv")

Rows: 540 Columns: 116
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (115): question, total_yes_percent, total_no_percent, total_doctor_refus...
dbl   (1): area

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 391 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): Sheet, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
# Go through each column name and clean it up (remove the total and percent tags)
clean_columns <- c()

for (col in colnames(scraped)) {  
  if (col == "area" | col == "question") {
    clean_columns <- c(clean_columns, col)
    next
  }
  if (startsWith(col, "total_")) {
    col <- substring(col, 7)
  }
  if (endsWith(col, "_percent")) {
    col <- substring(col, 1, nchar(col) - 8)
  }
  
  clean_columns <- c(clean_columns, col)
}

colnames(scraped) <- clean_columns      

In [6]:
# rotates the none area and question columns into a row called "response"
scraped <- scraped %>%
  pivot_longer(
    cols      = -c(area, question), 
    names_to  = "response",          
    values_to = "percent"           
  ) %>%
  filter(!is.na(percent))

In [7]:
clean <- function(x) {
  x <- gsub("[^A-Za-z0-9]+", "_", x)   # spaces/punct/newlines -> _
  gsub("^_+|_+$", "", x)               # trim leading/trailing _
}

In [8]:
scraped$variable <- paste(tolower(clean(scraped$question)), tolower(clean(scraped$response)), sep = "_")

In [9]:
brfss_features <- brfss[c("Sheet")]

cleaned_brfss <- unlist(lapply(brfss_features [[1]], function(x) tolower(clean(x))))

#scraped